# FEF Inactivation — exploring eye movements and neurons in Python

Welcome! This notebook walks you through a real monkey neurophysiology dataset:
eye movements and single-neuron recordings from the **frontal eye field (FEF)**,
recorded before, during and after the FEF was temporarily switched off with a drug.

**You do not need to know much Python to work through this.** Every cell is
commented, and the code deliberately uses simple, explicit `for` loops rather than
clever one-liners, so you can read it top to bottom and follow what happens.

By the end you will have:

1. Loaded a session and understood what is inside it
2. Aligned the eye traces to the moment the monkey was told to move
3. **Detected each saccade** (fast eye movement) and measured it
4. Looked at the recorded neuron as a raster and a PSTH
5. A list of open questions to explore yourself, with starter code

**How to run a cell:** click on it and press `Shift + Enter`. Run the cells in
order from the top the first time through — later cells depend on earlier ones.

---
# 1. What is this dataset?

## The big question

The **frontal eye field (FEF)** is a patch of cortex that helps control where you
look. If it really drives eye movements, then switching it off should change how
the monkey looks at things — and it should do so **only for one side of space**,
because the left FEF controls looking to the right and vice versa.

To test this, the experimenter injected a small amount of **muscimol** into the FEF.
Muscimol temporarily silences neurons in a small area; the effect builds up over
tens of minutes and wears off later. So a single recording session naturally splits
into trials **before**, **during**, and **after** the injection.

## The task, one trial at a time

Each trial follows the same script:

| Step | What the monkey sees | Event code |
|---|---|---|
| 1 | A dot appears in the middle. The monkey must look at it and hold still. | `SHOWFIXCD` (2), `FIXSTARTCD` (3) |
| 2 | A **fractal picture** appears off to one side (left or right). The monkey must keep staring at the centre dot — not look at it yet. | `TGTCD` (6) |
| 3 | About **467 ms later**, the centre dot disappears. This is the **GO cue** — now the monkey may look at the fractal. | `FIXOFF` (5) |
| 4 | The monkey makes a fast eye movement (a **saccade**) to the fractal and gets juice. | `RWDCD` (10) |

The **reaction time (RT)** is how long after the GO cue the saccade started.
That is the main behavioural measure in this notebook.

The fractal was one of two kinds: a **good object** (large reward) or a **bad
object** (small reward). So each trial has both a *direction* (left or right) and a
*value* (good or bad).

## Why we have to detect the saccades ourselves

You might expect the file to just tell us the reaction time. It does not. There is
an event code that *could* mark the saccade (`SAC_OCCURRED`), but it was only
recorded on about **11% of trials**. So the reaction time has to be measured from
the **eye position trace** itself. Section 7 is where we do that, and it is the
heart of this notebook.

## The three sessions

| Short name | Task | Trials | Target positions | Injection phases |
|---|---|---|---|---|
| `Adams102325_FRAC` | Fractal object directed saccade | 1740 | 20° up-left / down-right | before / during / after |
| `Adams110725_FRAC` | Fractal object directed saccade | 1337 | 15° left / right | before / during / after |
| `Adams110725_OneDR` | One-direction-rewarded | 470 | 15° left / right | before / during only |

Start with `Adams102325_FRAC`. Later, change one line and re-run everything on
another session — a good habit, and a good test of whether your analysis was
accidentally tuned to one session.

## One important caveat, before you compute any percentage

**Only the trials the monkey got right are stored in these files.** Trials where he
broke fixation, never fixated, or failed the saccade were counted elsewhere and
then dropped. For the 110725 session there were 2137 trials in total but only 1337
correct ones — so roughly 800 trials are simply missing from what you can see.

This matters most for the *anticipation rate*. The trials where the monkey jumped
the gun hardest are exactly the ones that became fixation breaks and got thrown
away. So any anticipation rate you measure here is an **underestimate**, and a
biased one. Keep that in mind rather than reporting the number as if it were
the truth.

---
# 2. Setup: packages and data

## The packages

We only use four, and all of them are already installed on Google Colab:

- **numpy** — arrays of numbers, and the maths to go with them. Always imported as `np`.
- **scipy** — extra scientific tools. We use exactly one function from it (a smoother).
- **matplotlib** — all the plotting. We use `matplotlib.pyplot`, always imported as `plt`.
- **pandas** — only to print tidy tables at the end. Imported as `pd`.

In [ ]:
# "import X as Y" means: load the toolbox X, and refer to it by the short name Y.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import savgol_filter   # the one scipy function we need

# Make the figures a comfortable size and reasonably readable.
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False     # drop the box around plots;
plt.rcParams["axes.spines.right"] = False   # it is just visual clutter

print("Packages loaded. numpy version:", np.__version__)

## Getting the data file

The data has been packed into a small `.npz` file (a `.npz` is simply a box holding
several numpy arrays). Each session is 6–24 MB, so downloading takes a few seconds.

**Choose your session here.** This is the only line you need to change to analyse a
different recording.

In [ ]:
# ============================================================================
# CHOOSE YOUR SESSION -- change this one line to analyse a different recording
# ============================================================================
SESSION = "Adams102325_FRAC"      # or "Adams110725_FRAC" or "Adams110725_OneDR"


# ---------------------------------------------------------------------------
# Where to get the file from.
#
# On Google Colab, leave USE_LOCAL_FILE as False. The data downloads itself.
#
# If you are running this notebook on your own computer and already have the
# .npz files, set USE_LOCAL_FILE = True and point LOCAL_FOLDER at them.
# ---------------------------------------------------------------------------
USE_LOCAL_FILE = False
LOCAL_FOLDER = "."

# Where the data files live. These are already filled in -- you should not need
# to change them.
RELEASE = "https://github.com/xuefeiyu2015/fef-inactivation-notebook/releases/download/data-v1"

DATA_URLS = {
    "Adams102325_FRAC": RELEASE + "/Adams102325_FRAC.npz",
    "Adams110725_FRAC": RELEASE + "/Adams110725_FRAC.npz",
    "Adams110725_OneDR": RELEASE + "/Adams110725_OneDR.npz",
}

In [ ]:
import os
import urllib.request

file_name = SESSION + ".npz"

if USE_LOCAL_FILE:
    data_path = os.path.join(LOCAL_FOLDER, file_name)

else:
    data_path = file_name

    if os.path.exists(data_path):
        print("Already downloaded:", data_path)
    else:
        print("Downloading", file_name, "...")
        urllib.request.urlretrieve(DATA_URLS[SESSION], data_path)
        print("Download finished.")

# Check the file really arrived before going any further.
if not os.path.exists(data_path):
    raise FileNotFoundError(
        "Could not find " + data_path + ".\n"
        "If you are on Colab, check that you are connected to the internet and\n"
        "run this cell again."
    )

size_mb = os.path.getsize(data_path) / 1e6
print("Data file ready:", data_path, "(%.1f MB)" % size_mb)

---
# 3. Loading the data and looking inside

`np.load` opens the `.npz` box. What comes back behaves like a **dictionary**: a
collection of named items, where you get an item by writing `box["name"]`.

Let us open it and list everything inside.

In [ ]:
session = np.load(data_path, allow_pickle=False)

print("This file contains", len(session.files), "items:\n")
for name in session.files:
    item = session[name]
    print("  %-22s shape %-16s type %s" % (name, str(item.shape), item.dtype))

### What those shapes mean

Look at the shapes above. Most items have a first number equal to the **number of
trials** — 1740 for the 102325 session. That is the key idea:

> **Row `i` of almost every array is trial `i`.**

So `target_x[5]` is the target position on trial 5, and `eye_x[5]` is the eye trace
on trial 5. Everything lines up, which is what makes it possible to ask questions
like "was the reaction time slower on the trials where the target was on the right?"

### A word about `nan`

You will see `nan` a lot. It stands for **"not a number"**, and it is how we write
*missing data*. It is not zero — zero is a real measurement, `nan` means we do not
know.

Two things to remember about `nan`:

1. Ordinary maths on `nan` gives `nan`. So `np.mean([1, 2, np.nan])` is `nan`.
   To ignore the missing values, use the `nan`-aware versions:
   **`np.nanmean`, `np.nanmedian`, `np.nansum`, `np.nanmax`**.
2. `nan` is not equal to anything, *including itself*. So `x == np.nan` is always
   False and never works. To test for it, use **`np.isnan(x)`**.

In [ ]:
# A quick demonstration, because this trips up nearly everyone at first.
values = np.array([1.0, 2.0, np.nan, 4.0])

print("np.mean    ->", np.mean(values), "   <- one missing value poisons the whole answer")
print("np.nanmean ->", np.nanmean(values), "   <- this is what you usually want")
print()
print("values == np.nan ->", values == np.nan, "  <- never use this, it is always False")
print("np.isnan(values) ->", np.isnan(values), "  <- use this instead")

---
# 4. The main variables

Here is everything in the file and what it means. **The single most important fact:**

> **Every time in this dataset is in milliseconds, counted from the start of that
> trial.** Event times, eye samples and spike times all use that same clock.

That is what makes the whole analysis possible: to ask "how long after the GO cue
did the spike happen?", you just *subtract* the GO cue time from the spike time.

### The eye traces

| Variable | Shape | Meaning |
|---|---|---|
| `eye_x` | trials × samples | Horizontal eye position, in **degrees of visual angle**. Positive = right. |
| `eye_y` | trials × samples | Vertical eye position, in degrees. Positive = up. |
| `eye_n_samples` | trials | How many real samples that trial has. |
| `eye_bin_width` | one number | Milliseconds between samples. It is **1**, i.e. 1000 samples per second. |

Trials have different lengths (roughly 3600–7300 samples). To store them in one
rectangular array, the short ones were padded at the end with `nan`. So the real
data of trial `i` is `eye_x[i, :eye_n_samples[i]]` — "row i, up to its real length".

Because the samples are 1 ms apart and start at the beginning of the trial,
**sample number = time in milliseconds**. Sample 500 happened 500 ms into the trial.

### The events

| Variable | Shape | Meaning |
|---|---|---|
| `event_codes` | trials × 18 | *What* happened, as numeric codes, in the order it happened. |
| `event_times` | trials × 18 | *When* each of those happened (ms from trial start). |

The two line up: if `event_codes[i, 3]` is `6`, then `event_times[i, 3]` is when
code `6` happened on trial `i`. Trials with fewer than 18 events are padded with `nan`.

### The target

| Variable | Shape | Meaning |
|---|---|---|
| `target_x`, `target_y` | trials | Where the fractal appeared, in degrees. |
| `target_angle` | trials | Same position as an angle. **0 = up, counting counter-clockwise**, so 90 = left, 180 = down, 270 = right. |
| `target_ecc` | trials | Distance from the centre, in degrees (15 or 20 here). |

⚠️ **Two traps worth knowing about**, both already handled for you:

- **Never test the angle with `==`.** The stored values are numbers like
  `45.00000002`, so `target_angle == 45` matches *nothing at all* and silently
  gives you an empty group. Use `target_x < 0` and `target_x > 0` instead — the
  sign of a number is safe to test.
- **Do not use the hemifield event codes to work out direction.** Each session
  used different codes for this (102/105 in one session, 80/81 in another), so code
  written around one session's numbers quietly produces empty groups on the next.
  `target_x` exists in every session and means the same thing every time.

### The injection

`injection_condition` has one value per trial:

| Value | Meaning |
|---|---|
| `0` | **before** the injection |
| `nan` | **during** the injection |
| `1` | **after** the injection |

Note that "during" is written as `nan`, so you must find those trials with
`np.isnan(injection_condition)` — not with `== np.nan`, which never works.

### The neuron

| Variable | Shape | Meaning |
|---|---|---|
| `spike_times` | trials × max spikes | The time of every spike, in ms from trial start, padded with `nan`. |

One neuron was recorded. Row `i` holds that trial's spike times; a trial with 35
spikes has 35 numbers followed by `nan` padding.

Let us now pull everything out of the box and into plain variables.

In [ ]:
# Pull each item out of the box and give it a short name we can use from now on.
eye_x = session["eye_x"]
eye_y = session["eye_y"]
eye_n_samples = session["eye_n_samples"]
eye_bin_width = float(session["eye_bin_width"])

event_codes = session["event_codes"]
event_times = session["event_times"]

target_x = session["target_x"]
target_y = session["target_y"]
target_angle = session["target_angle"]
target_ecc = session["target_ecc"]

injection_condition = session["injection_condition"]
spike_times = session["spike_times"]

# How many trials are there? The number of rows of any per-trial array.
n_trials = len(target_x)

print("Session:      ", str(session["session_name"]))
print("Task:         ", str(session["task_type"]), "(code %d)" % int(session["task_code"]))
print("Trials:       ", n_trials)
print("Eye sampling: ", "one sample every %.0f ms" % eye_bin_width)
print("Total spikes: ", int(np.sum(~np.isnan(spike_times))))

Let us look at one single trial, to make the shapes concrete.

In [ ]:
trial = 0     # Python counts from 0, so this is the FIRST trial

n_samples_here = eye_n_samples[trial]

print("Trial", trial)
print("  eye trace length:   %d samples = %d ms" % (n_samples_here, n_samples_here * eye_bin_width))
print("  target position:    x = %.1f deg, y = %.1f deg" % (target_x[trial], target_y[trial]))
print("  target angle:       %.2f deg   <- note it is not exactly a round number!" % target_angle[trial])
print("  eccentricity:       %.0f deg" % target_ecc[trial])
print()

# The eye position during the first 10 ms of the trial.
print("  first 10 eye_x samples:", np.round(eye_x[trial, :10], 2))
print()

# The spikes on this trial: drop the nan padding to see the real ones.
this_trial_spikes = spike_times[trial]
this_trial_spikes = this_trial_spikes[~np.isnan(this_trial_spikes)]
print("  number of spikes:   ", len(this_trial_spikes))
print("  first 5 spike times:", np.round(this_trial_spikes[:5], 1), "ms from trial start")

Notice the `~np.isnan(...)` in the cell above. That is the standard way to say
**"keep only the parts that are not missing"**:

- `np.isnan(x)` gives True where the value is missing
- `~` means **not**, so `~np.isnan(x)` is True where the value is real
- `x[~np.isnan(x)]` keeps only the real values

You will use this pattern constantly. It is worth reading that line until it feels
obvious.

---
# 5. Event codes: when did each thing happen?

The events are stored as numeric codes. Here are the ones that matter for us:

| Code | Name | Meaning |
|---|---|---|
| 2 | `SHOWFIXCD` | The central fixation dot appeared |
| 3 | `FIXSTARTCD` | The monkey started fixating it |
| 6 | `TGTCD` | **The fractal target appeared** |
| 5 | `FIXOFF` | **The GO cue** — fixation dot off, "you may look now" |
| 7 | `SAC_OCCURRED` | A saccade was detected online (only ~11% of trials — unreliable) |
| 10 | `RWDCD` | Reward delivered |
| 12 | `CORRECTCD` | Trial marked correct |
| 90 / 91 | `GOODOBJ` / `BADOBJ` | The fractal was the high-value / low-value object |

Notice that `FIXOFF` is code **5** and `TGTCD` is code **6** — the GO cue has the
*smaller* number even though it happens *later*. The codes are just labels, not an
order, so always look up the code by name rather than assuming.

To find *when* a code happened on each trial, we search that trial's row of
`event_codes` for the code, then read the matching entry of `event_times`.

In [ ]:
def find_event_time(event_codes, event_times, wanted_code):
    """Find when one event code happened, on every trial.

    Returns one number per trial: the time in ms from that trial's start.
    Trials where the code never occurred get nan.
    """
    n_trials = event_codes.shape[0]

    # Start with "missing everywhere", then fill in the trials where we find it.
    result = np.full(n_trials, np.nan)

    for i in range(n_trials):
        # Which positions in this trial's row hold the code we want?
        positions = np.where(event_codes[i] == wanted_code)[0]

        if len(positions) > 0:
            first_position = positions[0]      # if it happened twice, take the first
            result[i] = event_times[i, first_position]

    return result


# The two markers we care about most.
target_on_time = find_event_time(event_codes, event_times, 6)   # TGTCD
go_cue_time = find_event_time(event_codes, event_times, 5)      # FIXOFF

print("Target onset found on %d of %d trials" % (np.sum(~np.isnan(target_on_time)), n_trials))
print("GO cue found on       %d of %d trials" % (np.sum(~np.isnan(go_cue_time)), n_trials))

# How long did the monkey have to wait between seeing the target and being allowed
# to look at it?
delay = go_cue_time - target_on_time
print()
print("Delay from target onset to GO cue: %.0f +/- %.0f ms" % (np.nanmean(delay), np.nanstd(delay)))

**That delay is the key to a puzzle you will meet in section 8.** The wait is
almost always the same length — about 467 ms, varying by only ±9 ms. The monkey
did hundreds of these trials, so he learned the timing and started moving his eyes
*before* the GO cue actually appeared. That is why you will see some **negative
reaction times**, and they are real behaviour, not a bug.

Now let us check how often the online saccade marker was recorded — the reason we
have to detect saccades ourselves.

In [ ]:
def count_trials_with_code(event_codes, wanted_code):
    """Count how many trials contain a given event code at least once."""
    count = 0
    for i in range(event_codes.shape[0]):
        if np.any(event_codes[i] == wanted_code):
            count = count + 1
    return count


n_with_marker = count_trials_with_code(event_codes, 7)   # SAC_OCCURRED
print("SAC_OCCURRED was recorded on %d of %d trials (%.0f%%)"
      % (n_with_marker, n_trials, 100 * n_with_marker / n_trials))
print()
print("-> far too few to use. We will detect the saccades from the eye trace instead.")

## Sorting the trials into conditions

Now we build **masks**. A mask is an array of True/False, one entry per trial, that
says which trials belong to a group. You then use it to pick out those trials:
`reaction_time[is_left_saccade]` gives the reaction times of just the leftward trials.

In [ ]:
# --- Which way did the monkey have to look? ---
# Take this from the sign of the target's x position -- NOT from the hemifield
# event codes, which are different numbers in different sessions.
is_left_saccade = target_x < 0
is_right_saccade = target_x > 0

# --- Was the fractal the high-value or the low-value object? ---
def make_code_mask(event_codes, wanted_code):
    """True on every trial that contains the given event code."""
    n_trials = event_codes.shape[0]
    mask = np.zeros(n_trials, dtype=bool)      # start with all False
    for i in range(n_trials):
        if np.any(event_codes[i] == wanted_code):
            mask[i] = True
    return mask

is_good_object = make_code_mask(event_codes, 90)   # GOODOBJ, large reward
is_bad_object = make_code_mask(event_codes, 91)    # BADOBJ, small reward

print("Saccade direction:  %4d leftward, %4d rightward" % (np.sum(is_left_saccade), np.sum(is_right_saccade)))
print("Object value:       %4d good,     %4d bad" % (np.sum(is_good_object), np.sum(is_bad_object)))
print("Target eccentricity: %.0f deg" % np.nanmedian(target_ecc))

### The injection phases

Rather than assuming every session is "before / during / after", we **read the
phases out of the data**. Sessions differ, and code that assumes one layout breaks
silently on the next one — for example the OneDR session has no "after" phase at
all, because the recording ended while the drug was still active.

In [ ]:
def compute_injection_phases(injection_condition):
    """Work out which injection phases this session actually has.

    Returns a list of (name, mask) pairs, in the order before -> during -> after.
    """
    phases = []

    # "before" and "after" are ordinary numbers, "during" is nan, so it needs
    # np.isnan rather than a comparison.
    if np.any(injection_condition == 0):
        phases.append(("before", injection_condition == 0))

    if np.any(np.isnan(injection_condition)):
        phases.append(("during", np.isnan(injection_condition)))

    if np.any(injection_condition == 1):
        phases.append(("after", injection_condition == 1))

    # A session with a SECOND injection would carry the value 2. None of the three
    # sessions here does, but we handle it so nothing gets silently dropped.
    if np.any(injection_condition == 2):
        phases.append(("after 2nd injection", injection_condition == 2))

    return phases


injection_phases = compute_injection_phases(injection_condition)

print("This session has %d injection phases:" % len(injection_phases))
for name, mask in injection_phases:
    print("   %-20s %4d trials" % (name, np.sum(mask)))

# Safety check: every trial should belong to exactly one phase.
total_in_phases = 0
for name, mask in injection_phases:
    total_in_phases = total_in_phases + np.sum(mask)
print()
print("Trials covered by a phase: %d of %d" % (total_in_phases, n_trials))

---
# 6. Aligning the eye traces to the GO cue

## The problem

Every trial is recorded separately and has its own length, and the GO cue happens at
a **different moment in every trial** — around 2200 ms into the trial, but never at
exactly the same time. So sample 2200 means something different on every trial, and
you cannot compare or average the traces as they are.

## The fix

Cut a window around the GO cue on each trial, and place all those windows on **one
shared time axis where 0 means "the GO cue"**. After that, column `k` of the result
is always the same time relative to the GO cue, on every trial. Now the trials can
be compared, averaged and plotted together.

We will keep 300 ms before the GO cue and 600 ms after it.

## Why interpolation and not just cutting

The GO cue happened at, say, 2191.95 ms — not on a whole millisecond. But the eye
was only sampled at whole milliseconds (2191, 2192, ...). So to know where the eye
was at exactly 2191.95 ms, we read *between* two samples. That is **interpolation**,
and `np.interp` does it: given the known sample times and values, it estimates the
value at any time you ask for.

`np.interp` needs three things: the times you *want*, the times you *have*, and the
values you have. Anything outside the recorded range comes back as `nan`.

In [ ]:
def align_eye_traces(eye_x, eye_y, eye_n_samples, eye_bin_width,
                     align_time, ms_before, ms_after):
    """Cut a window out of every trial's eye trace, centred on a marker.

    align_time gives the marker time for each trial (ms from trial start).
    Returns aligned_x, aligned_y (trials x samples) and the shared time axis,
    on which 0 is the marker. Trials with no marker come back as all nan.
    """
    n_trials = len(align_time)

    # The shared time axis: -300, -299, ..., 0, ..., 599, 600 (in ms).
    time_axis = np.arange(-ms_before, ms_after + 1) * eye_bin_width

    # Start with everything missing, then fill trial by trial.
    aligned_x = np.full((n_trials, len(time_axis)), np.nan)
    aligned_y = np.full((n_trials, len(time_axis)), np.nan)

    for i in range(n_trials):
        if np.isnan(align_time[i]):
            continue           # no marker on this trial -> leave the row as nan

        # This trial's real data, with the nan padding removed.
        n_here = eye_n_samples[i]
        trial_x = eye_x[i, :n_here]
        trial_y = eye_y[i, :n_here]

        # When each of those samples was recorded, in ms from the trial start.
        # Samples are 1 ms apart, so this is simply 0, 1, 2, 3, ...
        sample_time = np.arange(n_here) * eye_bin_width

        # The moments we WANT to know about, on the trial's own clock.
        wanted_time = align_time[i] + time_axis

        # Read the trace at those moments. left/right say what to use when the
        # window reaches outside the recording: nan, i.e. "we do not know".
        aligned_x[i] = np.interp(wanted_time, sample_time, trial_x, left=np.nan, right=np.nan)
        aligned_y[i] = np.interp(wanted_time, sample_time, trial_y, left=np.nan, right=np.nan)

    return aligned_x, aligned_y, time_axis


MS_BEFORE = 300     # how much eye trace to keep before the GO cue
MS_AFTER = 600      # ...and after it

aligned_x, aligned_y, eye_time = align_eye_traces(
    eye_x, eye_y, eye_n_samples, eye_bin_width, go_cue_time, MS_BEFORE, MS_AFTER)

print("Aligned eye traces shape:", aligned_x.shape, " (trials x samples)")
print("Time axis runs from %.0f to %.0f ms, with 0 = the GO cue" % (eye_time[0], eye_time[-1]))

# A few trials are short enough that the window runs off the end of the recording.
# Those get some nan, which is honest -- we genuinely do not have that data.
n_incomplete = np.sum(np.any(np.isnan(aligned_x), axis=1))
print()
print("Trials with some missing samples in the window: %d of %d" % (n_incomplete, n_trials))

Let us look at what we have got. Below are the first 30 trials, split by which side
the target was on. Each thin line is one trial's horizontal eye position.

Before the GO cue (the dashed line at 0) the eye sits near the centre, at 0 degrees.
Shortly after, it jumps — up to positive values for rightward targets, down to
negative for leftward ones. **That jump is the saccade**, and measuring it is the
job of the next section.

In [ ]:
def plot_example_traces(time_axis, aligned_x, is_left, is_right, n_show=30):
    """Draw the horizontal eye position of a few trials, split by target side."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    groups = [("Leftward target", is_left, "tab:blue"),
              ("Rightward target", is_right, "tab:red")]

    for ax, (title, mask, colour) in zip(axes, groups):
        trials_to_show = np.where(mask)[0][:n_show]

        for i in trials_to_show:
            ax.plot(time_axis, aligned_x[i], color=colour, linewidth=0.7, alpha=0.6)

        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.text(5, ax.get_ylim()[1] * 0.9, "GO cue", fontsize=9)
        ax.set_title("%s  (%d trials shown)" % (title, len(trials_to_show)))
        ax.set_xlabel("Time from GO cue (ms)")

    axes[0].set_ylabel("Horizontal eye position (deg)")
    fig.suptitle("Raw aligned eye traces -- the saccade is the sudden jump", y=1.02)
    fig.tight_layout()
    plt.show()


plot_example_traces(eye_time, aligned_x, is_left_saccade, is_right_saccade)

---
# 7. Detecting the saccade

## The idea, in five steps

A saccade is a *fast* eye movement. So instead of looking at eye **position**, we
look at eye **speed** — how fast the eye is moving, in degrees per second. During
fixation the speed is near zero; during a saccade it shoots up to over 1000 deg/s.

1. **Smooth** the eye position a little, then turn it into **speed**.
2. Measure the **resting speed** in a quiet window before the GO cue.
3. A saccade is where speed rises **clearly above** that resting level and **stays**
   there for several samples in a row (one noisy sample is not enough).
4. Walk **backwards** from there to the moment the speed first crossed the
   threshold. That instant is the **saccade onset**, and its time is the **reaction time**.
5. Walk **forwards** past the peak to where the speed settles back down. That is the
   **saccade offset**.

## Step 1: position → speed

Two details matter here.

**Why smooth first?** The eye tracker is a little noisy. Speed is computed by
subtracting neighbouring samples, and subtracting neighbours *amplifies* noise
badly. Smoothing first keeps the speed trace readable. We use a Savitzky-Golay
filter, which smooths while preserving the sharp shape of a real saccade better
than a plain running average would.

**Handling missing data.** `savgol_filter` crashes if the trace contains any `nan`,
and some of our traces do at the edges. So the helper below smooths only the real
part of each trace and leaves the rest missing.

In [ ]:
def smooth_trace(trace, span=7, poly_order=2):
    """Smooth one trial's trace, coping with the nan padding at its edges.

    savgol_filter refuses to run on data containing nan, so we find the stretch of
    real data and smooth only that part.
    """
    smoothed = np.full(len(trace), np.nan)

    real_positions = np.where(~np.isnan(trace))[0]
    if len(real_positions) < span:
        return smoothed          # too little real data to smooth

    first = real_positions[0]
    last = real_positions[-1] + 1

    # If there is a gap of missing data in the MIDDLE, give up on this trial
    # rather than inventing values across the hole.
    if np.any(np.isnan(trace[first:last])):
        return smoothed

    smoothed[first:last] = savgol_filter(trace[first:last], span, poly_order)
    return smoothed


def compute_eye_speed(aligned_x, aligned_y, time_axis, smooth_span=7):
    """Smooth the eye position and convert it into speed in degrees per second.

    Returns the smoothed x and y, the speed, and the time axis the speed sits on.
    """
    n_trials = aligned_x.shape[0]

    smooth_x = np.full(aligned_x.shape, np.nan)
    smooth_y = np.full(aligned_y.shape, np.nan)

    for i in range(n_trials):
        smooth_x[i] = smooth_trace(aligned_x[i], smooth_span)
        smooth_y[i] = smooth_trace(aligned_y[i], smooth_span)

    # Time between two samples, in seconds (our samples are 1 ms apart).
    step_ms = np.median(np.diff(time_axis))
    step_seconds = step_ms / 1000.0

    # How far did the eye move between neighbouring samples, in x and in y?
    step_x = np.diff(smooth_x, axis=1)
    step_y = np.diff(smooth_y, axis=1)

    # Combine the two into a single distance with Pythagoras, then divide by the
    # time taken to get a speed. np.hypot(a, b) is sqrt(a*a + b*b).
    speed = np.hypot(step_x, step_y) / step_seconds

    # np.diff gives one fewer value than we started with, because each value
    # describes the gap BETWEEN two samples. So the speed sits at the midpoints.
    speed_time = (time_axis[:-1] + time_axis[1:]) / 2.0

    return smooth_x, smooth_y, speed, speed_time


smooth_x, smooth_y, eye_speed, speed_time = compute_eye_speed(aligned_x, aligned_y, eye_time)

print("Speed array shape:", eye_speed.shape)
print("Fastest speed seen anywhere: %.0f deg/s" % np.nanmax(eye_speed))

## Step 2: the settings

All the tuning lives in one place, so you can change one number and re-run. The
defaults below are the ones that were checked against the original MATLAB analysis.

In [ ]:
SETTINGS = {
    "smooth_span": 7,          # samples used by the smoother (about 7 ms)
    "baseline_window": (-200, -50),   # a quiet stretch, used to measure resting speed
    "search_window": (-50, 400),      # only look for a saccade onset in here
    "velocity_threshold": 40,  # deg/s ABOVE resting speed to count as moving
    "n_contiguous": 5,         # samples the speed must stay above, in a row
    "max_duration_ms": 200,    # a saccade cannot last longer than this
    "min_amplitude_deg": 3,    # anything smaller is drift, not a saccade
    "noise_speed": 2500,       # faster than this is a blink or a tracker glitch
    "offset_window_factor": 2,    # the offset is judged over a longer window...
    "offset_min_fraction": 0.5,   # ...of which only half needs to be below threshold
}

### Two settings worth understanding

**`noise_speed` (the blink guard).** Blinks and tracker glitches produce
impossibly fast "movements", so anything above this speed is thrown out. The trap
is that this number must sit **above the fastest real saccade**, or it quietly
deletes good data instead of bad. Real 20° saccades here peak at 1200–1500 deg/s,
and the original analysis found that setting this to 1500 threw away **a quarter of
the good trials**, while 2500 keeps 98.7% of them and still catches genuine blinks.
If you ever change the smoothing or use a recording with a different sampling rate,
check this number again.

**`search_window` starts at −50 ms, before the GO cue.** That is deliberate.
Remember the delay was almost always 467 ms — the monkey learned it and sometimes
started moving *before* the GO cue. If we only looked after 0, we would miss those
trials entirely and bias our reaction times.

## Step 3: finding the saccade

Two small helper functions do the searching. They are worth reading, because the
difference between them is the whole reason the detector works well.

In [ ]:
def find_first_run(flags, search_positions, n_needed):
    """Find the first place where flags is True n_needed times IN A ROW.

    Requiring a run, rather than a single sample, is what stops one noisy sample
    from being mistaken for a saccade. Returns None if there is no such run.
    """
    last_possible_start = len(flags) - n_needed

    for position in search_positions:
        if position < 0 or position > last_possible_start:
            continue
        if np.all(flags[position:position + n_needed]):
            return position

    return None


def find_first_majority(flags, search_positions, window_length, n_needed):
    """Find the first window of window_length samples containing at least
    n_needed True values. Returns the START of that window, or None.

    This is the relaxed cousin of find_first_run: that one asks "are they ALL
    true?", this one asks "are ENOUGH of them true?".
    """
    last_possible_start = len(flags) - window_length

    for position in search_positions:
        if position < 0 or position > last_possible_start:
            continue
        if np.sum(flags[position:position + window_length]) >= n_needed:
            return position

    return None

**Why two different rules?** The *start* of a saccade is clean: the speed rises
through the threshold and stays up. So `find_first_run` — all samples above — works
well there.

The *end* is messy. The speed undershoots, wobbles, and a small corrective movement
often pushes a sample or two back above the threshold. Demanding that *every* sample
be below the threshold makes the search walk straight past the real landing point
and report the saccade as longer than it was. So the offset uses the looser
`find_first_majority` rule: half of a 10-sample window below the threshold is
enough. In the original analysis this shortened the median duration from 44 ms to a
much more realistic 38 ms, without changing the onset at all.

One more subtlety: **the search for the offset starts after the speed PEAK**, not
right after the onset. On the way up, the speed is still climbing through the
threshold, so some samples are below it — a "half the window" rule could be
satisfied there and end the saccade before it began. A saccade's end must come
after its fastest moment, so that is where we start looking.

Now the detector itself.

In [ ]:
def detect_saccades(smooth_x, smooth_y, speed, speed_time, settings):
    """Find and measure the saccade on every trial.

    Returns a dictionary of arrays, one entry per trial, in trial order.
    Undetected trials are nan everywhere, so every array still lines up with
    the other per-trial variables.
    """
    n_trials = speed.shape[0]

    # Prepare the answers: everything missing until we fill it in.
    detected = np.zeros(n_trials, dtype=bool)
    reaction_time = np.full(n_trials, np.nan)
    saccade_end = np.full(n_trials, np.nan)
    amplitude = np.full(n_trials, np.nan)
    angle = np.full(n_trials, np.nan)
    peak_velocity = np.full(n_trials, np.nan)
    start_x = np.full(n_trials, np.nan)
    start_y = np.full(n_trials, np.nan)
    end_x = np.full(n_trials, np.nan)
    end_y = np.full(n_trials, np.nan)
    onset_index = np.full(n_trials, np.nan)
    offset_index = np.full(n_trials, np.nan)

    # Mark impossibly fast samples as noise (blinks, tracker glitches). They are
    # set to nan, so they can never trigger a detection.
    is_noise = speed > settings["noise_speed"]
    speed = np.where(is_noise, np.nan, speed)

    # Which samples fall in the baseline window, and which in the search window?
    # These are the same for every trial, so work them out once.
    base_lo, base_hi = settings["baseline_window"]
    in_baseline = (speed_time >= base_lo) & (speed_time <= base_hi)

    search_lo, search_hi = settings["search_window"]
    search_positions = np.where((speed_time >= search_lo) & (speed_time <= search_hi))[0]

    step_ms = np.median(np.diff(speed_time))
    max_duration_samples = int(round(settings["max_duration_ms"] / step_ms))

    # The offset rule: a window this long, this many of which must be below.
    offset_window = int(round(settings["n_contiguous"] * settings["offset_window_factor"]))
    offset_n_needed = max(1, int(round(offset_window * settings["offset_min_fraction"])))

    for i in range(n_trials):
        trial_speed = speed[i]

        # --- the resting speed on this trial, and the threshold from it ---
        baseline_samples = trial_speed[in_baseline]
        if np.all(np.isnan(baseline_samples)):
            continue                          # no usable baseline on this trial
        baseline = np.nanmean(baseline_samples)
        threshold = baseline + settings["velocity_threshold"]

        # Note these are kept separate on purpose: a nan (noise) sample is neither
        # above nor below, so a glitch can neither start nor end a saccade.
        is_above = trial_speed > threshold
        is_below = trial_speed < threshold

        # --- the onset ---
        trigger = find_first_run(is_above, search_positions, settings["n_contiguous"])
        if trigger is None:
            continue                          # no saccade found on this trial

        # The trigger is where we became SURE. The true onset is a little earlier,
        # where the speed first crossed the threshold on its way up.
        onset = trigger
        while onset > 0 and trial_speed[onset - 1] >= threshold:
            onset = onset - 1

        # --- the offset ---
        last_allowed = min(onset + max_duration_samples, len(trial_speed) - 1)

        stretch = trial_speed[onset:last_allowed + 1]
        if np.all(np.isnan(stretch)):
            continue
        peak = onset + int(np.nanargmax(stretch))     # where the speed peaked

        offset = find_first_majority(is_below, range(peak + 1, last_allowed + 1),
                                     offset_window, offset_n_needed)
        if offset is None:
            offset = last_allowed        # still moving at the limit: stop there

        # --- reject the trial if a blink landed inside the saccade ---
        if np.any(is_noise[i, onset:offset + 1]):
            continue

        # --- measure it ---
        # Speed sample k describes the gap between position samples k and k+1,
        # so the movement runs from position onset to position offset+1.
        from_x = smooth_x[i, onset]
        from_y = smooth_y[i, onset]
        to_x = smooth_x[i, offset + 1]
        to_y = smooth_y[i, offset + 1]

        if np.isnan(from_x) or np.isnan(to_x):
            continue

        this_amplitude = np.hypot(to_x - from_x, to_y - from_y)
        if this_amplitude < settings["min_amplitude_deg"]:
            continue                          # drift, not a saccade

        detected[i] = True
        reaction_time[i] = speed_time[onset]
        saccade_end[i] = speed_time[offset]
        amplitude[i] = this_amplitude
        peak_velocity[i] = np.nanmax(trial_speed[onset:offset + 1])
        start_x[i] = from_x
        start_y[i] = from_y
        end_x[i] = to_x
        end_y[i] = to_y
        onset_index[i] = onset
        offset_index[i] = offset

        # The direction the eye actually travelled, written in the SAME convention
        # as target_angle (0 = up, counter-clockwise). np.arctan2 gives the usual
        # maths convention (0 = right), so the +270 and the wrap line them up.
        direction = np.degrees(np.arctan2(to_y - from_y, to_x - from_x))
        angle[i] = np.mod(direction + 270, 360)

    return {
        "detected": detected,
        "reaction_time": reaction_time,
        "saccade_end": saccade_end,
        "duration": saccade_end - reaction_time,
        "amplitude": amplitude,
        "angle": angle,
        "peak_velocity": peak_velocity,
        "start_x": start_x,
        "start_y": start_y,
        "end_x": end_x,
        "end_y": end_y,
        "onset_index": onset_index,
        "offset_index": offset_index,
    }


saccades = detect_saccades(smooth_x, smooth_y, eye_speed, speed_time, SETTINGS)

n_detected = np.sum(saccades["detected"])
print("Saccade detected on %d of %d trials (%.1f%%)" % (n_detected, n_trials, 100 * n_detected / n_trials))

## Did it work? Look at the traces before you trust any number

This is the most important figure in the notebook. Numbers from a detector you have
not looked at are worthless. Each panel below is one trial:

- the **top** row shows eye position (x in blue, y in orange)
- the **bottom** row shows eye speed, with the trial's threshold as a dashed line
- the **green line** is where the detector put the saccade **onset**
- the **red line** is where it put the **offset**

The onset should land right where the position starts to move and the speed starts
to shoot up. The offset should land where the eye arrives and the speed collapses.

In [ ]:
def plot_detection_check(time_axis, speed_time, smooth_x, smooth_y, speed,
                         saccades, settings, trials_to_show):
    """Draw position and speed for a few trials, with the detected saccade marked.

    This function only draws what it is handed -- it does no detecting of its own,
    so what you see is exactly what the detector reported.
    """
    n_show = len(trials_to_show)
    fig, axes = plt.subplots(2, n_show, figsize=(3.4 * n_show, 6), sharex=True)

    for column, i in enumerate(trials_to_show):
        ax_pos = axes[0, column]
        ax_speed = axes[1, column]

        ax_pos.plot(time_axis, smooth_x[i], color="tab:blue", label="eye x")
        ax_pos.plot(time_axis, smooth_y[i], color="tab:orange", label="eye y")
        ax_speed.plot(speed_time, speed[i], color="black", linewidth=1)

        # The onset and offset the detector found on this trial.
        if saccades["detected"][i]:
            onset_ms = saccades["reaction_time"][i]
            offset_ms = saccades["saccade_end"][i]
            for ax in (ax_pos, ax_speed):
                ax.axvline(onset_ms, color="tab:green", linewidth=1.5)
                ax.axvline(offset_ms, color="tab:red", linewidth=1.5)

            title = "trial %d\nRT %.0f ms, %.1f deg" % (i, onset_ms, saccades["amplitude"][i])
        else:
            title = "trial %d\nNOT DETECTED" % i

        # The speed threshold used on this trial.
        onset = saccades["onset_index"][i]
        if not np.isnan(onset):
            baseline_lo, baseline_hi = settings["baseline_window"]
            in_baseline = (speed_time >= baseline_lo) & (speed_time <= baseline_hi)
            threshold = np.nanmean(speed[i][in_baseline]) + settings["velocity_threshold"]
            ax_speed.axhline(threshold, color="grey", linestyle="--", linewidth=1)

        for ax in (ax_pos, ax_speed):
            ax.axvline(0, color="black", linestyle=":", linewidth=1)

        ax_pos.set_title(title, fontsize=10)
        ax_speed.set_xlabel("Time from GO cue (ms)")

    axes[0, 0].set_ylabel("Eye position (deg)")
    axes[1, 0].set_ylabel("Eye speed (deg/s)")
    axes[0, 0].legend(fontsize=8, loc="upper left")

    fig.suptitle("Detection check: green = saccade onset, red = offset, "
                 "dotted = GO cue, dashed grey = speed threshold", y=1.01)
    fig.tight_layout()
    plt.show()


# Show the first six trials where something was detected.
detected_trials = np.where(saccades["detected"])[0]
plot_detection_check(eye_time, speed_time, smooth_x, smooth_y, eye_speed,
                     saccades, SETTINGS, detected_trials[:6])

Now the same thing for many trials at once. Each line is one trial's eye speed,
shifted so that **time 0 is that trial's own detected saccade onset** rather than
the GO cue. If the detector is working, all the speed peaks should stack up
neatly just after 0 instead of being smeared out.

In [ ]:
def compute_speed_aligned_to_saccade(speed, speed_time, saccades, ms_before=100, ms_after=150):
    """Re-cut the speed traces so that 0 is each trial's own saccade onset."""
    new_time = np.arange(-ms_before, ms_after + 1)
    n_trials = speed.shape[0]
    out = np.full((n_trials, len(new_time)), np.nan)

    for i in range(n_trials):
        if not saccades["detected"][i]:
            continue
        wanted = saccades["reaction_time"][i] + new_time
        out[i] = np.interp(wanted, speed_time, speed[i], left=np.nan, right=np.nan)

    return out, new_time


def plot_speed_overlay(speed_aligned, time_axis, n_show=100):
    """Overlay the saccade-aligned speed of many trials, plus their average."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    shown = 0
    for i in range(speed_aligned.shape[0]):
        if np.all(np.isnan(speed_aligned[i])):
            continue
        ax.plot(time_axis, speed_aligned[i], color="grey", linewidth=0.4, alpha=0.35)
        shown = shown + 1
        if shown >= n_show:
            break

    ax.plot(time_axis, np.nanmean(speed_aligned, axis=0), color="tab:red",
            linewidth=2.5, label="average of all detected trials")
    ax.axvline(0, color="tab:green", linewidth=1.5, label="detected saccade onset")

    ax.set_xlabel("Time from saccade onset (ms)")
    ax.set_ylabel("Eye speed (deg/s)")
    ax.set_title("%d individual trials, aligned on their own detected onset" % shown)
    ax.legend()
    fig.tight_layout()
    plt.show()


speed_aligned, saccade_time = compute_speed_aligned_to_saccade(eye_speed, speed_time, saccades)
plot_speed_overlay(speed_aligned, saccade_time)

---
# 8. The saccade parameters: what each number means

The detector returned one number per trial for each of these. Undetected trials are
`nan` throughout, so the arrays still line up with everything else.

| Parameter | Units | What it means |
|---|---|---|
| `reaction_time` | ms | **When the saccade started**, measured from the GO cue. The headline behavioural measure. Small = quick. |
| `saccade_end` | ms | When the eye landed, from the GO cue. |
| `duration` | ms | `saccade_end − reaction_time`. How long the movement took. Typically 27–55 ms here. |
| `amplitude` | degrees | **How far the eye travelled**, start point to end point. Should match the target eccentricity (15 or 20°). |
| `angle` | degrees | **Which way it travelled.** Same convention as `target_angle`: 0 = up, counter-clockwise, so 90 = left, 270 = right. |
| `peak_velocity` | deg/s | **The fastest the eye moved** during the saccade. Typically 1000–1500 deg/s. |
| `start_x`, `start_y` | degrees | Where the eye was when the saccade began — near the centre. |
| `end_x`, `end_y` | degrees | **Where the eye landed.** Compare with `target_x`, `target_y` for accuracy. |
| `detected` | True/False | Whether a saccade was found at all. Always check this first. |

## Three things about these numbers that surprise people

**1. Negative reaction times are real.** The delay between the target and the GO cue
was almost always 467 ms, so the monkey learned it and often started moving before
the GO cue formally arrived. A negative RT means "he jumped the gun". These are
genuine behaviour and should not be deleted — though remember that the trials where
he jumped *hardest* became fixation breaks and were dropped before the file was
written, so what you see is the mild end of the anticipation.

**2. RT and peak velocity can move independently.** They measure different things —
*when* the movement started versus *how fast* it went — and inactivating the FEF can
affect one without the other. Do not assume that "slower" means the same thing for both.

**3. Amplitude is the best single check on the detector.** The monkey was looking at
a target at a known distance. If the median amplitude comes out near the target
eccentricity, the detector is finding the real, target-directed saccade rather than
a blink or a small correction.

Let us look at the distributions.

In [ ]:
def plot_parameter_distributions(saccades, target_ecc):
    """Histograms of the four main saccade measures."""
    ok = saccades["detected"]

    panels = [
        ("reaction_time", "Reaction time (ms)", None),
        ("amplitude", "Amplitude (deg)", np.nanmedian(target_ecc)),
        ("peak_velocity", "Peak velocity (deg/s)", None),
        ("duration", "Duration (ms)", None),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))

    for ax, (key, label, reference) in zip(axes, panels):
        values = saccades[key][ok]
        ax.hist(values, bins=50, color="tab:blue", edgecolor="none")

        median = np.nanmedian(values)
        ax.axvline(median, color="black", linewidth=1.5)
        ax.set_title("%s\nmedian %.0f" % (label, median), fontsize=10)
        ax.set_xlabel(label)

        # For amplitude, also draw where the target actually was.
        if reference is not None:
            ax.axvline(reference, color="tab:red", linestyle="--", linewidth=1.5)
            ax.set_title("%s\nmedian %.1f (target at %.0f)" % (label, median, reference), fontsize=10)

    axes[0].set_ylabel("Number of trials")
    # Mark where "anticipated the GO cue" begins.
    axes[0].axvline(0, color="tab:red", linestyle="--", linewidth=1.5)
    fig.suptitle("Distributions of the saccade measures (red dashed = reference)", y=1.06)
    fig.tight_layout()
    plt.show()


plot_parameter_distributions(saccades, target_ecc)

n_anticipated = np.sum(saccades["reaction_time"][saccades["detected"]] < 0)
print("Saccades starting BEFORE the GO cue: %d of %d detected (%.0f%%)"
      % (n_anticipated, np.sum(saccades["detected"]),
         100 * n_anticipated / np.sum(saccades["detected"])))

## Check 1: did the saccades actually go towards the target?

This is the strongest single test that the detector found the *right* movement. We
compare the direction the eye travelled (`angle`) against where the target actually
was (`target_angle`).

The `np.mod(difference + 180, 360) - 180` below wraps the difference into the range
−180 to +180. Without it, a saccade at 5° and a target at 355° would look 350°
apart when they are really only 10° apart.

In [ ]:
def compute_angle_error(saccade_angle, target_angle):
    """How far off the target direction was each saccade, in degrees (-180..180)."""
    difference = saccade_angle - target_angle
    return np.mod(difference + 180, 360) - 180


angle_error = compute_angle_error(saccades["angle"], target_angle)

ok = saccades["detected"]
n_close = np.sum(np.abs(angle_error[ok]) < 30)

print("Median direction error: %.1f deg" % np.nanmedian(np.abs(angle_error[ok])))
print("Saccades within 30 deg of the target: %d of %d (%.1f%%)"
      % (n_close, np.sum(ok), 100 * n_close / np.sum(ok)))
print()
print("-> if that percentage is near 100, the detector is finding the real,")
print("   target-directed saccade and not a blink or a small correction.")

## Check 2: where did the eye actually land?

Each dot below is one trial's landing point. The crosses are the two target
positions. The dots should cluster tightly on the crosses.

In [ ]:
def plot_landing_points(saccades, target_x, target_y, is_left, is_right):
    """Scatter of where each saccade ended, with the true target positions marked."""
    fig, ax = plt.subplots(figsize=(6, 6))

    ok = saccades["detected"]

    groups = [("Leftward target", is_left & ok, "tab:blue"),
              ("Rightward target", is_right & ok, "tab:red")]

    for label, mask, colour in groups:
        ax.plot(saccades["end_x"][mask], saccades["end_y"][mask], ".",
                color=colour, markersize=3, alpha=0.3, label=label)

        # The true target position for this group.
        ax.plot(np.median(target_x[mask]), np.median(target_y[mask]), "X",
                color="black", markersize=14, markeredgecolor="white", markeredgewidth=1.5)

    ax.plot(0, 0, "+", color="black", markersize=14)   # the fixation point
    ax.set_xlabel("Horizontal eye position (deg)")
    ax.set_ylabel("Vertical eye position (deg)")
    ax.set_title("Where each saccade landed\n(X = true target, + = fixation point)")
    ax.axis("equal")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_landing_points(saccades, target_x, target_y, is_left_saccade, is_right_saccade)

## Check 3: the main sequence

The **main sequence** is one of the most reliable facts about saccades: bigger
saccades are faster, following a tight, slightly curved relationship. Every healthy
oculomotor system shows it.

If your detected saccades fall on a clean curve here, they are real saccades. If
the plot is a shapeless cloud, the detector is picking up noise.

In [ ]:
def plot_main_sequence(saccades):
    """Peak velocity against amplitude -- the classic saccade main sequence."""
    ok = saccades["detected"]

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(saccades["amplitude"][ok], saccades["peak_velocity"][ok], ".",
            color="tab:purple", markersize=3, alpha=0.3)

    ax.set_xlabel("Amplitude (deg)")
    ax.set_ylabel("Peak velocity (deg/s)")
    ax.set_title("The main sequence: bigger saccades are faster")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()


plot_main_sequence(saccades)

## Putting it together: a summary table

Now the payoff. We split the reaction times by **saccade direction** and by
**injection phase**, and see what the drug did.

Remember the anatomy: the left FEF drives saccades to the **right**, and vice versa.
So if the injection worked, it should affect **one direction only** — the
*contralateral* one. The other direction acts as a built-in control: if both
directions changed equally, that would point to the monkey getting tired rather than
to the drug.

In [ ]:
def compute_bootstrap_ci(values, n_resamples=1000, low_pct=2.5, high_pct=97.5):
    """How uncertain is a median? Returns the lower and upper edge of its range.

    The idea, called bootstrapping, is simple: pretend our trials are the whole
    world, draw a fresh sample of the same size FROM them (allowing repeats), and
    take the median of that. Do it a thousand times and you get a thousand
    plausible medians. The middle 95% of those is the range the true median
    plausibly lies in.
    """
    values = values[~np.isnan(values)]
    if len(values) < 3:
        return np.nan, np.nan

    medians = np.full(n_resamples, np.nan)
    for k in range(n_resamples):
        picked = np.random.choice(values, size=len(values), replace=True)
        medians[k] = np.median(picked)

    return np.percentile(medians, low_pct), np.percentile(medians, high_pct)


def compute_summary_by_phase(values, valid, phases, groups):
    """Median of a per-trial measure, split by phase and by group.

    phases and groups are both lists of (name, mask) pairs, so this one function
    can split the data any way you like -- by injection phase and direction, by
    phase and object value, by early/late trials, and so on.

    Returns a plain list of rows. Computation only: no printing, no plotting.
    """
    rows = []

    for phase_name, phase_mask in phases:
        for group_name, group_mask in groups:
            selected = phase_mask & group_mask & valid
            chosen = values[selected]

            if np.any(selected):
                middle = np.nanmedian(chosen)
                low, high = compute_bootstrap_ci(chosen)
            else:
                middle, low, high = np.nan, np.nan, np.nan

            rows.append({"phase": phase_name, "group": group_name,
                         "n": int(np.sum(selected)),
                         "value": middle, "low": low, "high": high})

    return rows


def compute_proportion_by_phase(flag, valid, phases, groups):
    """Percentage of trials where flag is True, split by phase and group.

    Returns rows in the SAME shape as compute_summary_by_phase, so the same
    plotting function draws either one.
    """
    rows = []

    for phase_name, phase_mask in phases:
        for group_name, group_mask in groups:
            selected = phase_mask & group_mask & valid
            n = int(np.sum(selected))

            if n > 0:
                n_true = int(np.sum(flag & selected))
                percent = 100.0 * n_true / n
                # The uncertainty of a percentage, from the same bootstrap idea.
                spread = 100.0 * np.sqrt((percent / 100) * (1 - percent / 100) / n)
                low, high = percent - 1.96 * spread, percent + 1.96 * spread
            else:
                percent, low, high = np.nan, np.nan, np.nan

            rows.append({"phase": phase_name, "group": group_name, "n": n,
                         "value": percent, "low": low, "high": high})

    return rows


# The two ways we most often split the trials. Each is a list of (name, mask).
DIRECTION_GROUPS = [("leftward", is_left_saccade), ("rightward", is_right_saccade)]
VALUE_GROUPS = [("good object", is_good_object), ("bad object", is_bad_object)]

ALL_TRIALS = np.ones(n_trials, dtype=bool)   # useful when nothing should be excluded

Now the plotting workhorse. It takes whatever `compute_summary_by_phase` or
`compute_proportion_by_phase` returned and draws it as a grouped bar chart with
error bars. **Almost every question in section 10 is answered with this one
function**, so it is worth a look.

The error bars matter. A difference between two bars only means something if the
bars' error ranges do not overlap much — otherwise you are reading noise.

In [ ]:
def plot_summary_by_phase(rows, y_label, title, colours=None):
    """Grouped bar chart of the rows produced by the two compute_ functions above.

    Draws only what it is handed -- it computes nothing itself.
    """
    # Work out the phases and the groups, keeping the order they appear in.
    phase_names = []
    group_names = []
    for row in rows:
        if row["phase"] not in phase_names:
            phase_names.append(row["phase"])
        if row["group"] not in group_names:
            group_names.append(row["group"])

    if colours is None:
        colours = ["tab:blue", "tab:red", "tab:green", "tab:orange"]

    fig, ax = plt.subplots(figsize=(1.9 * len(phase_names) + 4, 4.5))

    bar_width = 0.8 / len(group_names)
    positions = np.arange(len(phase_names))

    for g, group_name in enumerate(group_names):
        heights, down, up, labels = [], [], [], []

        for phase_name in phase_names:
            # Find the one row matching this phase and this group.
            for row in rows:
                if row["phase"] == phase_name and row["group"] == group_name:
                    heights.append(row["value"])
                    # Error bars are given as distances from the top of the bar.
                    down.append(0 if np.isnan(row["low"]) else row["value"] - row["low"])
                    up.append(0 if np.isnan(row["high"]) else row["high"] - row["value"])
                    labels.append(row["n"])

        offset = (g - (len(group_names) - 1) / 2) * bar_width
        bars = ax.bar(positions + offset, heights, bar_width,
                      yerr=[down, up], capsize=4,
                      color=colours[g % len(colours)], label=group_name)

        # Write the value ABOVE the error bar (not on the bar, where the whisker
        # would strike through it), and the trial count inside the bar.
        for bar, height, up_error, n in zip(bars, heights, up, labels):
            if np.isnan(height):
                continue
            ax.text(bar.get_x() + bar.get_width() / 2, height + up_error,
                    "%.0f" % height, ha="center", va="bottom", fontsize=9)
            ax.text(bar.get_x() + bar.get_width() / 2, 0, "n=%d\n" % n,
                    ha="center", va="bottom", fontsize=7, color="white")

    # Leave headroom at the top so the legend has somewhere to sit that is not
    # on top of the bars.
    all_tops = [row["value"] + (0 if np.isnan(row["high"]) else row["high"] - row["value"])
                for row in rows if not np.isnan(row["value"])]
    if all_tops:
        highest = max(all_tops)
        lowest = min(0, min(row["value"] for row in rows if not np.isnan(row["value"])))
        ax.set_ylim(lowest * 1.1 if lowest < 0 else 0, highest * 1.3)

    ax.set_xticks(positions)
    ax.set_xticklabels(phase_names)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


# Reaction time, split by injection phase and saccade direction.
rt_rows = compute_summary_by_phase(saccades["reaction_time"], saccades["detected"],
                                   injection_phases, DIRECTION_GROUPS)

plot_summary_by_phase(rt_rows, "Median reaction time (ms)",
                      "Reaction time by injection phase and saccade direction")

Look at the bars before reading on. The leftward bars barely move; the rightward
one jumps during the injection and comes back down. The error bars on the "before"
and "during" bars do not overlap, so that jump is bigger than the noise.

Here are the other three measures, drawn the same way.

In [ ]:
for measure_name, unit in [("peak_velocity", "deg/s"), ("amplitude", "deg"), ("duration", "ms")]:
    rows = compute_summary_by_phase(saccades[measure_name], saccades["detected"],
                                    injection_phases, DIRECTION_GROUPS)
    plot_summary_by_phase(rows, "Median %s (%s)" % (measure_name.replace("_", " "), unit),
                          "%s by injection phase and direction" % measure_name.replace("_", " ").capitalize())

If you also want the numbers as a table, `pandas` will lay the same rows out for
you. We use it only for printing — never to do the analysis.

In [ ]:
table = pd.DataFrame(rt_rows).pivot(index="group", columns="phase", values="value")
table = table[[name for name, mask in injection_phases]]     # keep them in time order

print("MEDIAN REACTION TIME (ms)")
print(table.round(0))

### How to read those figures

On the `Adams102325_FRAC` session the bars should show this:

- **Reaction time**: leftward saccades barely change (98 → 102 → 104 ms), while
  rightward ones slow down sharply during the injection and then recover
  (116 → 152 → 114 ms).
- **Peak velocity**: leftward barely changes, while rightward drops from about
  1480 to **951 deg/s** after the injection — and stays down.

Two things are worth noticing.

**The effect is one-sided.** Only rightward saccades change. That is exactly what an
injection into one hemisphere should do, and it is much stronger evidence than an
overall slowing would be.

**RT and peak velocity dissociate.** RT slows *during* and recovers *after*; peak
velocity barely moves *during* and collapses *after*. That looks odd, but three
independent numbers in the figures above say the slow saccades are real: the
amplitude is unchanged, the duration rises to match (which is exactly what "same
distance, lower speed" must mean), and the detection rate stays high in every phase,
so it is not a case of the detector quietly failing on some trials.

## Seeing it trial by trial

Averages can hide things. This figure shows every trial in the order it was
recorded, with a running median through it, and the injection phases shaded. You can
watch the effect build up and wear off.

In [ ]:
def compute_running_median(trial_numbers, values, mask, window):
    """A running median of a measure over trials, within one group of trials.

    Returns the trial numbers and the smoothed values, for plotting.
    """
    group_trials = trial_numbers[mask]
    group_values = values[mask]

    smoothed = np.full(len(group_values), np.nan)

    for k in range(len(group_values)):
        lo = max(0, k - window // 2)
        hi = min(len(group_values), k + window // 2 + 1)
        chunk = group_values[lo:hi]
        if not np.all(np.isnan(chunk)):
            smoothed[k] = np.nanmedian(chunk)

    return group_trials, smoothed


def plot_session_course(trial_numbers, values, groups, phases, y_label, title, window=25):
    """Draw a measure across the whole session, one line per group, phases shaded."""
    fig, ax = plt.subplots(figsize=(12, 4.5))

    # Draw the data FIRST, so that the y axis is scaled to it before we try to
    # put the phase labels along the top.
    for label, mask, colour in groups:
        # The raw trials, faint.
        ax.plot(trial_numbers[mask], values[mask], ".", color=colour,
                markersize=2, alpha=0.18, zorder=1)
        # The running median, bold.
        x, y = compute_running_median(trial_numbers, values, mask, window)
        ax.plot(x, y, color=colour, linewidth=2, label=label, zorder=2)

    # Make a little empty space at the top of the plot, so the phase names have
    # somewhere to sit without landing on the data or the title.
    y_low, y_high = ax.get_ylim()
    height = y_high - y_low
    ax.set_ylim(y_low, y_high + 0.12 * height)
    label_y = y_high + 0.09 * height

    # Now shade the injection phases behind everything, and name each one.
    shades = {"before": "white", "during": "#ffe8b0", "after": "#cfe6ff",
              "after 2nd injection": "#d8f0d0"}
    for phase_name, phase_mask in phases:
        phase_trials = trial_numbers[phase_mask]
        ax.axvspan(phase_trials.min(), phase_trials.max(),
                   color=shades.get(phase_name, "#eeeeee"), alpha=0.7, zorder=0)
        ax.text(np.median(phase_trials), label_y, phase_name,
                ha="center", va="center", fontsize=10)

    ax.set_xlabel("Trial number (in the order they were recorded)")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()


trial_numbers = np.arange(n_trials)
ok = saccades["detected"]

direction_groups = [("Leftward saccade", is_left_saccade & ok, "tab:blue"),
                    ("Rightward saccade", is_right_saccade & ok, "tab:red")]

plot_session_course(trial_numbers, saccades["reaction_time"], direction_groups,
                    injection_phases, "Reaction time (ms)",
                    "Reaction time across the session (running median of %d trials)" % 25)

plot_session_course(trial_numbers, saccades["peak_velocity"], direction_groups,
                    injection_phases, "Peak velocity (deg/s)",
                    "Peak velocity across the session (running median of %d trials)" % 25)

---
# 9. The neuron: rasters and PSTHs

One FEF neuron was recorded throughout. `spike_times[i]` holds the times of every
spike on trial `i`, in ms from that trial's start, padded with `nan`.

Two standard ways of looking at a neuron:

- A **raster** draws one row per trial and one tick per spike. You see the raw data.
- A **PSTH** (peri-stimulus time histogram) counts the spikes into small time bins
  and averages over trials, giving firing rate in spikes per second. You see the
  pattern.

Both need the spikes **aligned** to an event first — exactly the same subtraction
idea as for the eye traces.

## Why we align to two different events

We will align the same spikes twice: once to **target onset**, once to the **saccade
onset**. Comparing the two answers a real question about what this neuron does.

- If the neuron responds to *seeing* the target, its response is locked to target
  onset. Aligned to the target it looks sharp; aligned to the saccade it smears out,
  because the reaction time varies from trial to trial.
- If the neuron drives the *movement*, the opposite happens.

So putting the two side by side separates a **visual** response from a **motor** one.

In [ ]:
def align_spikes(spike_times, align_time, window, bin_width=1):
    """Re-express every spike as time from a marker, and count them into bins.

    Returns:
      bin_centres   the time axis, in ms from the marker
      counts        trials x bins, the number of spikes in each bin
      per_trial     a list, one entry per trial, of that trial's aligned spike times
                    (this is what a raster plot needs)
    """
    n_trials = spike_times.shape[0]

    edges = np.arange(window[0], window[1] + bin_width, bin_width)
    bin_centres = edges[:-1] + bin_width / 2.0

    counts = np.full((n_trials, len(bin_centres)), np.nan)
    per_trial = []

    for i in range(n_trials):
        if np.isnan(align_time[i]):
            per_trial.append(np.array([]))
            continue          # no marker: leave this row as nan, NOT as zero

        # This trial's spikes, padding removed, shifted so 0 is the marker.
        trial_spikes = spike_times[i]
        trial_spikes = trial_spikes[~np.isnan(trial_spikes)] - align_time[i]

        # Keep only the ones inside our window, for the raster.
        in_window = (trial_spikes >= window[0]) & (trial_spikes < window[1])
        per_trial.append(trial_spikes[in_window])

        counts[i] = np.histogram(trial_spikes, edges)[0]

    return bin_centres, counts, per_trial


def compute_psth(bin_centres, counts, mask, bin_width=1, smooth_ms=20):
    """Average firing rate over a group of trials, in spikes per second."""
    group_counts = counts[mask]

    # Mean spikes per bin, then convert to spikes per second.
    mean_per_bin = np.nanmean(group_counts, axis=0)
    rate = mean_per_bin * (1000.0 / bin_width)

    # Smooth it a little, otherwise 1 ms bins are far too spiky to read.
    kernel = np.ones(smooth_ms) / smooth_ms
    return np.convolve(rate, kernel, mode="same")

An important detail is hidden in `align_spikes`: a trial with no marker gets a row
of `nan`, **not a row of zeros**. "We could not measure this trial" and "this trial
had no spikes" are completely different statements, and writing zero would quietly
drag the average down.

Now the figure. Two columns — target-aligned on the left, saccade-aligned on the
right — with the raster above and the PSTH below.

In [ ]:
def plot_spike_overview(alignments, is_left, is_right, n_raster_trials=150):
    """Raster and PSTH, one column per alignment.

    alignments is a list of (name, bin_centres, counts, per_trial) tuples.
    """
    n_columns = len(alignments)
    fig, axes = plt.subplots(2, n_columns, figsize=(6.5 * n_columns, 7),
                             gridspec_kw={"height_ratios": [2, 1]})

    groups = [("Leftward", is_left, "tab:blue"), ("Rightward", is_right, "tab:red")]

    for column, (name, bin_centres, counts, per_trial) in enumerate(alignments):
        ax_raster = axes[0, column]
        ax_psth = axes[1, column]

        # --- the raster: one row of dots per trial, grouped by direction ---
        row = 0
        for label, mask, colour in groups:
            trials_here = np.where(mask)[0][:n_raster_trials]
            for i in trials_here:
                spikes = per_trial[i]
                if len(spikes) > 0:
                    ax_raster.plot(spikes, np.full(len(spikes), row), "|",
                                   color=colour, markersize=2.5, markeredgewidth=0.6)
                row = row + 1
            # A line separating the two direction groups.
            ax_raster.axhline(row, color="black", linewidth=0.8)

        ax_raster.set_ylim(row, -1)
        ax_raster.set_ylabel("Trial (grouped by direction)")
        ax_raster.set_title("Aligned to %s" % name)
        ax_raster.axvline(0, color="black", linestyle="--", linewidth=1.2)

        # --- the PSTH below ---
        for label, mask, colour in groups:
            rate = compute_psth(bin_centres, counts, mask)
            ax_psth.plot(bin_centres, rate, color=colour, linewidth=1.8, label=label)

        ax_psth.axvline(0, color="black", linestyle="--", linewidth=1.2)
        ax_psth.set_xlabel("Time from %s (ms)" % name)
        ax_psth.set_ylabel("Firing rate (spikes/s)")
        ax_psth.legend(fontsize=9)

    # Give both PSTHs the same y axis, so the columns can be compared fairly.
    psth_axes = [axes[1, c] for c in range(n_columns)]
    top = max(ax.get_ylim()[1] for ax in psth_axes)
    for ax in psth_axes:
        ax.set_ylim(0, top)

    fig.tight_layout()
    plt.show()


# The saccade onset in TRIAL time: the RT is measured from the GO cue, so adding
# the GO cue time puts it back on the trial's own clock.
saccade_onset_time = go_cue_time + saccades["reaction_time"]

target_centres, target_counts, target_spikes = align_spikes(
    spike_times, target_on_time, window=(-300, 600))

saccade_centres, saccade_counts, saccade_spikes = align_spikes(
    spike_times, saccade_onset_time, window=(-300, 600))

plot_spike_overview(
    [("target onset", target_centres, target_counts, target_spikes),
     ("saccade onset", saccade_centres, saccade_counts, saccade_spikes)],
    is_left_saccade, is_right_saccade)

## Measuring the firing rate in a window

To put numbers on it, count the spikes in a chosen window and divide by how long the
window was. Two windows are standard here:

- a **baseline** window before the event, to say what the neuron does at rest
- a **response** window just after it, to catch the reaction

⚠️ **A firing rate is meaningless unless you say what it was aligned to.** A
"baseline" of −200 to 0 ms before the *target* is genuine rest; the same window
before the *GO cue* falls in the middle of the delay period, when the neuron is
already responding to the target it can see. On this session those two give
5.2 and 15.7 spikes/s — a threefold difference from the same window name. Always
state the alignment.

In [ ]:
def compute_firing_rate(bin_centres, counts, window, bin_width=1):
    """Firing rate of every trial inside a time window, in spikes per second.

    Trials with no marker (an all-nan row) return nan, so they drop out of an
    average instead of counting as a zero-spike trial.
    """
    in_window = (bin_centres >= window[0]) & (bin_centres < window[1])

    n_spikes = np.nansum(counts[:, in_window], axis=1)

    # Only count bins that actually hold data.
    n_valid_bins = np.sum(~np.isnan(counts[:, in_window]), axis=1)
    seconds_observed = n_valid_bins * bin_width / 1000.0

    rate = np.full(counts.shape[0], np.nan)
    measured = n_valid_bins > 0
    rate[measured] = n_spikes[measured] / seconds_observed[measured]
    return rate


baseline_rate = compute_firing_rate(target_centres, target_counts, (-200, 0))
visual_rate = compute_firing_rate(target_centres, target_counts, (50, 250))

print("Firing rate, aligned to TARGET ONSET")
print("  baseline (-200 to 0 ms):  %.1f spikes/s" % np.nanmean(baseline_rate))
print("  response (50 to 250 ms):  %.1f spikes/s" % np.nanmean(visual_rate))
print()

print("Baseline firing rate by injection phase (target-aligned, -200 to 0 ms):")
for phase_name, phase_mask in injection_phases:
    print("   %-20s %.1f spikes/s" % (phase_name, np.nanmean(baseline_rate[phase_mask])))

---
# 10. What can you explore with this dataset?

You now have everything you need: per-trial saccade measures, trial conditions, and
a recorded neuron, all lined up row by row. Below are questions worth asking,
roughly easiest first.

**Every question starts with a figure**, because a plot shows you the size of an
effect and the spread around it, while a printed number hides both. Each cell
draws something and then leaves `TODO` lines for you to take further.

## Your toolkit

Everything below is already in memory:

| Name | What it is |
|---|---|
| `saccades["reaction_time"]`, `["peak_velocity"]`, `["amplitude"]`, `["duration"]`, `["end_x"]`, `["end_y"]`, `["detected"]` | per-trial saccade measures |
| `DIRECTION_GROUPS`, `VALUE_GROUPS` | ready-made `(name, mask)` group lists |
| `is_left_saccade`, `is_right_saccade`, `is_good_object`, `is_bad_object` | the individual masks |
| `injection_phases` | list of `(name, mask)` pairs |
| `ALL_TRIALS` | a mask that selects everything |
| `trial_numbers` | 0, 1, 2, ... in recording order |
| `baseline_rate`, `visual_rate` | per-trial firing rates |
| `target_counts`, `target_centres` | spikes aligned to target onset |

The two functions you will use most:

```python
rows = compute_summary_by_phase(values, valid, phases, groups)   # medians + error bars
plot_summary_by_phase(rows, "y axis label", "title")             # draws them
```

Both `phases` and `groups` are just lists of `(name, mask)` pairs, so you can split
the data **any way you like** — not only by injection phase. Q2 uses that to
compare early against late trials.

**Two habits worth keeping.** Always pass `saccades["detected"]` as `valid`, so
undetected trials cannot sneak in. And always use `np.nanmedian` / `np.nanmean`
rather than the plain versions.

---
## Level 1 — warm-up

### Q1. Does the value of the object change the reaction time?

The fractal was either a high-value or a low-value object. Monkeys usually look
faster at things worth more. Does this one? And — the more interesting half —
**does inactivating the FEF change the size of that value effect?**

The figure splits by object value instead of direction, which is a one-word change
to the call: pass `VALUE_GROUPS` where the earlier cells passed `DIRECTION_GROUPS`.

In [ ]:
# --- Q1 ------------------------------------------------------------------------
value_rows = compute_summary_by_phase(saccades["reaction_time"], saccades["detected"],
                                      injection_phases, VALUE_GROUPS)

plot_summary_by_phase(value_rows, "Median reaction time (ms)",
                      "Q1: does object value change the reaction time?")


# The bars show medians. The full distributions show whether the two conditions
# really differ, or just happen to have slightly different middles.
def plot_distributions_by_group(values, valid, groups, x_label, title, bins=45):
    """Overlaid histograms, one per group, drawn as outlines so they can overlap."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    colours = ["tab:blue", "tab:red", "tab:green", "tab:orange"]
    for g, (group_name, group_mask) in enumerate(groups):
        chosen = values[group_mask & valid]
        chosen = chosen[~np.isnan(chosen)]
        ax.hist(chosen, bins=bins, histtype="step", linewidth=2,
                color=colours[g % len(colours)],
                label="%s (median %.0f)" % (group_name, np.median(chosen)))
        ax.axvline(np.median(chosen), color=colours[g % len(colours)],
                   linestyle="--", linewidth=1)

    ax.set_xlabel(x_label)
    ax.set_ylabel("Number of trials")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_distributions_by_group(saccades["reaction_time"], saccades["detected"], VALUE_GROUPS,
                            "Reaction time (ms)", "Q1: the full RT distributions")

# TODO 1: is the value effect the same in every injection phase? The bars are
#         already split that way -- read them across, and say whether the gap
#         between the two colours changes.
# TODO 2: the two object types were not evenly spread across the two directions.
#         Redo the value comparison for rightward trials ONLY, by passing
#         [("good", is_good_object & is_right_saccade),
#          ("bad", is_bad_object & is_right_saccade)]
#         as the groups. Does the effect survive?

### Q2. Is the effect really the drug, or just the monkey getting tired?

A monkey that has done 1500 trials is not the same monkey that started. Tiredness
would slow **both** directions; a one-sided injection should slow only one.

Notice the trick in the cell below: `phases` does not have to be the injection
phases. Any list of `(name, mask)` pairs works, so we hand it early and late blocks
of trials instead.

In [ ]:
# --- Q2 ------------------------------------------------------------------------
# Build our own "phases": the first 100 and the last 100 trials of the session.
time_blocks = [
    ("first 100 trials", trial_numbers < 100),
    ("last 100 trials", trial_numbers >= n_trials - 100),
]

drift_rows = compute_summary_by_phase(saccades["reaction_time"], saccades["detected"],
                                      time_blocks, DIRECTION_GROUPS)

plot_summary_by_phase(drift_rows, "Median reaction time (ms)",
                      "Q2: start of the session versus the end")

# The session-course figure from section 8 is the other half of the answer --
# it shows the whole shape rather than just two ends.
plot_session_course(trial_numbers, saccades["reaction_time"], direction_groups,
                    injection_phases, "Reaction time (ms)",
                    "Q2: the whole session, for comparison")

# TODO 1: how big is the ipsilateral (barely-affected) drift from start to end?
#         Anything the drug did has to be clearly bigger than THAT to be believable.
# TODO 2: try splitting into four blocks instead of two, so you can see whether
#         the drift is steady or all happens at one point.

### Q3. Could the effect be the detector quietly failing?

If the detector found fewer saccades during the injection, the "slower" reaction
times might just be the surviving trials, not a real change. The detection rate
should stay high (97–100%) everywhere.

This is a proportion rather than a median, so it uses `compute_proportion_by_phase`
— which returns rows in the same shape, so the same plotting function draws it.

In [ ]:
# --- Q3 ------------------------------------------------------------------------
detection_rows = compute_proportion_by_phase(saccades["detected"], ALL_TRIALS,
                                             injection_phases, DIRECTION_GROUPS)

plot_summary_by_phase(detection_rows, "Saccades detected (%)",
                      "Q3: detection rate -- it should be high everywhere")

# TODO: does any bar drop noticeably below the others? If one falls under about
# 95%, look at those trials with plot_detection_check before trusting that
# group's numbers anywhere else in the notebook.

---
## Level 2 — the main findings

### Q4. Why do reaction time and peak velocity behave differently?

On the 102325 session, RT slows *during* the injection and recovers *after*, while
peak velocity barely moves *during* and drops sharply *after*. Two different time
courses from the same drug.

The clearest way to see it is to put every measure on **one** axis by expressing
each as a percentage of its own "before" value. Then a flat line means "unchanged"
and the shapes can be compared directly even though the units are different.

In [ ]:
# --- Q4 ------------------------------------------------------------------------
def plot_relative_change(measure_names, valid, phases, group, title):
    """Each measure as a percentage of its own value in the first phase.

    Putting everything on a common scale is what makes measures with different
    units (ms, deg/s, deg) comparable on one axis.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    phase_labels = [name for name, mask in phases]

    for measure_name in measure_names:
        rows = compute_summary_by_phase(saccades[measure_name], valid, phases, [group])
        values = np.array([row["value"] for row in rows])

        # A measure whose starting value is near zero cannot be turned into a
        # sensible percentage, so skip it rather than print nonsense.
        if abs(values[0]) < 1:
            print("skipping %s: its 'before' value is too close to zero" % measure_name)
            continue

        ax.plot(range(len(values)), 100 * values / values[0], "o-", linewidth=2,
                markersize=8, label=measure_name.replace("_", " "))

    ax.axhline(100, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(range(len(phase_labels)))
    ax.set_xticklabels(phase_labels)
    ax.set_ylabel("Percentage of the 'before' value (%)")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_relative_change(["reaction_time", "peak_velocity", "amplitude", "duration"],
                     saccades["detected"], injection_phases,
                     ("rightward", is_right_saccade),
                     "Q4: every measure relative to its own 'before' value (rightward)")

# TODO 1: run the same plot for ("leftward", is_left_saccade). The lines should be
#         much flatter. That contrast IS the result.
# TODO 2: if peak velocity drops by about 35% while amplitude stays put, duration
#         has to rise to compensate. Does it rise by roughly the right amount?

### Q5. Did the saccades become less accurate?

Speed is not the only thing that can break. Two different questions hide here:
did the eye land **further from the target on average**, and did the landing points
become **more scattered**? A movement can get worse in either way, or both.

In [ ]:
# --- Q5 ------------------------------------------------------------------------
# How far the eye landed from where the target actually was.
landing_error = np.hypot(saccades["end_x"] - target_x, saccades["end_y"] - target_y)

error_rows = compute_summary_by_phase(landing_error, saccades["detected"],
                                      injection_phases, DIRECTION_GROUPS)
plot_summary_by_phase(error_rows, "Median landing error (deg)",
                      "Q5: how far from the target did the eye land?")


def plot_landing_by_phase(saccades, target_x, target_y, phases, direction_mask, title):
    """One scatter of landing points per injection phase, side by side."""
    fig, axes = plt.subplots(1, len(phases), figsize=(4.2 * len(phases), 4.4),
                             sharex=True, sharey=True)
    if len(phases) == 1:
        axes = [axes]

    for ax, (phase_name, phase_mask) in zip(axes, phases):
        selected = phase_mask & direction_mask & saccades["detected"]

        ax.plot(saccades["end_x"][selected], saccades["end_y"][selected], ".",
                color="tab:red", markersize=4, alpha=0.35)
        ax.plot(np.median(target_x[selected]), np.median(target_y[selected]), "X",
                color="black", markersize=14, markeredgecolor="white", markeredgewidth=1.5)
        ax.plot(0, 0, "+", color="black", markersize=12)

        spread = np.nanstd(saccades["end_x"][selected])
        ax.set_title("%s\nspread %.2f deg (n=%d)" % (phase_name, spread, np.sum(selected)),
                     fontsize=10)
        ax.set_xlabel("Horizontal (deg)")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("Vertical (deg)")
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()


plot_landing_by_phase(saccades, target_x, target_y, injection_phases,
                      is_right_saccade, "Q5: where rightward saccades landed, by phase")

# TODO 1: run the same figure for is_left_saccade and compare the spreads.
# TODO 2: "spread" above is only the horizontal scatter. Add the vertical spread
#         too -- a saccade can become imprecise in a direction the target never
#         varied in, which is a different kind of failure.

### Q6. How often did the monkey jump the gun?

A negative reaction time means the saccade started before the GO cue.

**Then be careful with the answer.** Only correct trials are in this file. The
trials where the monkey anticipated hardest broke fixation and were thrown away
before the file was written, so every anticipation rate here is an underestimate —
and possibly an unevenly biased one, if he anticipated more in one phase than
another. Say so when you report the number.

In [ ]:
# --- Q6 ------------------------------------------------------------------------
is_anticipated = saccades["reaction_time"] < 0

anticipation_rows = compute_proportion_by_phase(is_anticipated, saccades["detected"],
                                                injection_phases, DIRECTION_GROUPS)

plot_summary_by_phase(anticipation_rows, "Saccades started before the GO cue (%)",
                      "Q6: anticipation rate (an UNDERESTIMATE -- see the text)")

# TODO 1: does the monkey anticipate more in the direction the drug affected, or
#         the other one? What would each answer mean?
# TODO 2: "anticipated" is currently RT < 0. Try RT < 50 ms instead -- a saccade
#         50 ms after the cue is too fast to be a genuine reaction either. Does
#         the picture change?

### Q7. Does the main sequence itself shift?

The main sequence (peak velocity against amplitude) is a property of the
oculomotor plant and its drive. If inactivation only made the monkey *start* later,
the main sequence should be unchanged. If it weakened the drive to the muscles, the
whole curve should drop.

Raw scatter clouds overlap too much to judge by eye, so the second figure bins the
saccades by amplitude and plots the median velocity in each bin — the honest way to
compare two clouds.

In [ ]:
# --- Q7 ------------------------------------------------------------------------
def compute_binned_median(x_values, y_values, edges):
    """Median of y within each bin of x. Returns bin centres and medians.

    Bins holding fewer than 5 points return nan, because a median of three
    points is not worth plotting.
    """
    centres = (edges[:-1] + edges[1:]) / 2
    medians = np.full(len(centres), np.nan)

    for k in range(len(centres)):
        in_bin = (x_values >= edges[k]) & (x_values < edges[k + 1])
        if np.sum(in_bin) >= 5:
            medians[k] = np.nanmedian(y_values[in_bin])

    return centres, medians


fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
colours = ["tab:green", "tab:orange", "tab:purple", "tab:brown"]
edges = np.arange(5, 30, 1.0)

for (phase_name, phase_mask), colour in zip(injection_phases, colours):
    selected = phase_mask & is_right_saccade & saccades["detected"]

    # Left panel: the raw clouds.
    axes[0].plot(saccades["amplitude"][selected], saccades["peak_velocity"][selected],
                 ".", color=colour, markersize=3, alpha=0.3, label=phase_name)

    # Right panel: the same data, binned by amplitude.
    centres, medians = compute_binned_median(saccades["amplitude"][selected],
                                             saccades["peak_velocity"][selected], edges)
    axes[1].plot(centres, medians, "o-", color=colour, linewidth=2, label=phase_name)

axes[0].set_title("Raw scatter -- too overlapping to judge")
axes[1].set_title("Binned medians -- now you can compare")
for ax in axes:
    ax.set_xlabel("Amplitude (deg)")
    ax.set_ylabel("Peak velocity (deg/s)")
    ax.legend()
fig.suptitle("Q7: main sequence by injection phase (rightward saccades)", y=1.03)
fig.tight_layout()
plt.show()

# TODO 1: repeat for is_left_saccade. Do the binned lines lie on top of each other
#         there, as they should if the drug only affected one side?
# TODO 2: at a FIXED amplitude, how much lower is the "after" line? That is the
#         cleanest single number for "the movement itself got weaker".

---
## Level 3 — the neuron

### Q8. Is this neuron visual, motor, or both?

If the neuron responds to *seeing* the target, its response is locked to target
onset: sharp when aligned to the target, smeared when aligned to the saccade. If it
drives the *movement*, the opposite. Overlaying the two PSTHs on one axis settles it.

In [ ]:
# --- Q8 ------------------------------------------------------------------------
def plot_psth_comparison(alignments, mask, title):
    """Overlay PSTHs from several alignments on one axis, to compare sharpness."""
    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    for name, centres, counts, colour in alignments:
        rate = compute_psth(centres, counts, mask)
        peak_height = np.max(rate[centres >= 0])
        peak_time = centres[centres >= 0][np.argmax(rate[centres >= 0])]

        ax.plot(centres, rate, color=colour, linewidth=2,
                label="%s (peak %.0f sp/s at %.0f ms)" % (name, peak_height, peak_time))
        ax.plot(peak_time, peak_height, "o", color=colour, markersize=9)

    ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax.set_xlabel("Time from the aligned event (ms)")
    ax.set_ylabel("Firing rate (spikes/s)")
    ax.set_title(title)
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()


plot_psth_comparison(
    [("target onset", target_centres, target_counts, "tab:blue"),
     ("saccade onset", saccade_centres, saccade_counts, "tab:red")],
    ALL_TRIALS,
    "Q8: the same spikes, aligned two ways -- which is sharper?")

# TODO 1: a tall, narrow, early peak after the TARGET with only a broad hump at
#         the saccade means a VISUAL neuron. Which is this one?
# TODO 2: try the other sessions. The 110725 recording has a much more strongly
#         visual unit than 102325 does.

### Q9. Is the neuron direction-selective, and does the drug silence it?

An FEF neuron usually prefers one direction. The sharper question is whether the
neuron's firing changes across the injection phases **in the same way the behaviour
does**.

If muscimol silenced the neuron you recorded, its rate should drop. If the
behavioural effect appears while this neuron is unchanged, that tells you the
injection affected tissue beyond your electrode — a real result, not a failure.

In [ ]:
# --- Q9 ------------------------------------------------------------------------
rate_rows = compute_summary_by_phase(visual_rate, ALL_TRIALS,
                                     injection_phases, DIRECTION_GROUPS)

plot_summary_by_phase(rate_rows, "Firing rate (spikes/s)",
                      "Q9: response to the target (50-250 ms), by phase and direction")

# The PSTHs behind those bars, one line per phase, preferred direction only.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for (phase_name, phase_mask), colour in zip(injection_phases,
                                            ["tab:green", "tab:orange", "tab:purple"]):
    rate = compute_psth(target_centres, target_counts, phase_mask & is_right_saccade)
    ax.plot(target_centres, rate, color=colour, linewidth=2, label=phase_name)

ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
ax.set_xlabel("Time from target onset (ms)")
ax.set_ylabel("Firing rate (spikes/s)")
ax.set_title("Q9: the neuron's response across the injection, rightward trials")
ax.legend()
plt.show()

# TODO 1: does the neuron prefer one direction, and is that preference the same
#         in every phase?
# TODO 2: does its rate go DOWN after the injection, as a silencing drug implies?
#         Check the 110725 session too -- it may not do what you expect, and that
#         is worth thinking about carefully rather than explaining away.

### Q10. Does the neuron predict the reaction time, trial by trial?

The most ambitious question here. Both `baseline_rate` and
`saccades["reaction_time"]` are one number per trial, so they can be compared
directly. A classic finding is that **higher pre-movement activity goes with faster
reaction times**.

A scatter of 150 noisy points is hard to read, so the figure also bins the trials by
firing rate and plots the median RT in each bin. If there is a trend, the binned
line shows it; if the line is flat, there is nothing there.

Test it *within one phase and one direction at a time*. Pooling across phases can
manufacture a correlation out of both measures drifting over the session.

In [ ]:
# --- Q10 -----------------------------------------------------------------------
# The delay period: after the target appeared, before the GO cue.
delay_rate = compute_firing_rate(target_centres, target_counts, (200, 400))

fig, axes = plt.subplots(1, len(injection_phases),
                         figsize=(4.6 * len(injection_phases), 4.4), sharey=True)
if len(injection_phases) == 1:
    axes = [axes]

for ax, (phase_name, phase_mask) in zip(axes, injection_phases):
    selected = phase_mask & is_right_saccade & saccades["detected"]

    x = delay_rate[selected]
    y = saccades["reaction_time"][selected]

    # Drop trials where either measure is missing, or the correlation is nan.
    both_present = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[both_present], y[both_present]

    ax.plot(x, y, ".", color="tab:grey", markersize=4, alpha=0.4)

    # The binned trend line, which is what you should actually judge.
    edges = np.linspace(np.min(x), np.max(x), 7)
    centres, medians = compute_binned_median(x, y, edges)
    ax.plot(centres, medians, "o-", color="tab:red", linewidth=2)

    r = np.corrcoef(x, y)[0, 1] if len(x) > 2 else np.nan
    ax.set_title("%s\nr = %.3f  (n = %d)" % (phase_name, r, len(x)), fontsize=10)
    ax.set_xlabel("Delay firing rate (spikes/s)")

axes[0].set_ylabel("Reaction time (ms)")
fig.suptitle("Q10: does the neuron predict the reaction time? (rightward trials)", y=1.03)
fig.tight_layout()
plt.show()

# TODO 1: is the red line sloped in the same direction in every phase, or does it
#         wander? A real effect should be consistent.
# TODO 2: is a correlation of this size meaningful on this many trials? Try
#         scipy.stats.pearsonr(x, y), which also gives a p-value.
# TODO 3: try other windows instead of (200, 400). If the answer depends on which
#         window you pick, how much should you trust it?

---
## Level 4 — the bigger questions

### Q11. Do the three sessions agree?

A result that appears in one recording is a hint; one that repeats across recordings
is a finding. The cell below wraps the whole notebook's pipeline into a single
function and runs it on all three sessions, so they can be compared on one figure.

Reading `analyse_session` is also a good check on your own understanding — every
line in it is something you have already done step by step above.

*This cell downloads the other two sessions and re-runs the detection, so give it
a minute.*

In [ ]:
# --- Q11 -----------------------------------------------------------------------
def analyse_session(session_name):
    """Run the whole pipeline on one session and return what we need to compare.

    Every step below is one you have already seen in sections 5 to 7.
    """
    path = session_name + ".npz"
    if not os.path.exists(path):
        print("  downloading", session_name, "...")
        urllib.request.urlretrieve(DATA_URLS[session_name], path)

    box = np.load(path)

    go_cue = find_event_time(box["event_codes"], box["event_times"], 5)

    a_x, a_y, t_axis = align_eye_traces(box["eye_x"], box["eye_y"], box["eye_n_samples"],
                                        float(box["eye_bin_width"]), go_cue,
                                        MS_BEFORE, MS_AFTER)
    s_x, s_y, spd, spd_t = compute_eye_speed(a_x, a_y, t_axis)
    found = detect_saccades(s_x, s_y, spd, spd_t, SETTINGS)

    target = box["target_x"]
    return {
        "saccades": found,
        "groups": [("leftward", target < 0), ("rightward", target > 0)],
        "phases": compute_injection_phases(box["injection_condition"]),
        "n_trials": len(target),
    }


ALL_SESSIONS = ["Adams102325_FRAC", "Adams110725_FRAC", "Adams110725_OneDR"]

results = {}
for name in ALL_SESSIONS:
    print("Analysing", name)
    results[name] = analyse_session(name)
    print("   %d trials, %d saccades detected"
          % (results[name]["n_trials"], np.sum(results[name]["saccades"]["detected"])))

In [ ]:
# One panel per session, so the shapes can be compared even though the sessions
# have different numbers of phases (OneDR has no "after").
def plot_across_sessions(results, measure_name, y_label, title):
    """Same measure, same layout, one panel per session."""
    fig, axes = plt.subplots(1, len(results), figsize=(5.2 * len(results), 4.4))
    if len(results) == 1:
        axes = [axes]

    for ax, (session_name, result) in zip(axes, results.items()):
        found = result["saccades"]
        phase_labels = [name for name, mask in result["phases"]]

        for (group_name, group_mask), colour in zip(result["groups"], ["tab:blue", "tab:red"]):
            values = []
            for phase_name, phase_mask in result["phases"]:
                selected = phase_mask & group_mask & found["detected"]
                values.append(np.nanmedian(found[measure_name][selected]))
            ax.plot(range(len(values)), values, "o-", color=colour,
                    linewidth=2, markersize=8, label=group_name)

        ax.set_xticks(range(len(phase_labels)))
        ax.set_xticklabels(phase_labels)
        ax.set_title(session_name, fontsize=10)
        ax.set_xlabel("Injection phase")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel(y_label)
    axes[0].legend()
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    plt.show()


plot_across_sessions(results, "reaction_time", "Median reaction time (ms)",
                     "Q11: does the pattern repeat across sessions?")

plot_across_sessions(results, "peak_velocity", "Median peak velocity (deg/s)",
                     "Q11: peak velocity across sessions")

# TODO 1: does the contralateral-only pattern hold in all three? Where it does
#         not, what is different about that session?
# TODO 2: the two FRAC sessions used different target positions (20 deg diagonal
#         versus 15 deg horizontal). Could eccentricity explain any difference in
#         the size of the effect?
# TODO 3: on Adams110725_FRAC the leftward RT after the injection is close to
#         ZERO -- the monkey moves as the cue arrives. Before believing it, plot
#         those trials with plot_detection_check and look at them.

### Q12. What is different about the OneDR session?

In the One-Direction-Rewarded task, **only one direction is rewarded** in a given
block, rather than value being carried by the object. That session also has no
"after" phase — the recording ended while the drug was still active, which is why
its panel above has two points instead of three.

- Does the reward *direction*, rather than the object, drive the reaction times?
- Does the inactivation interact with which direction was worth something?
- Handling a session with only two phases without special-casing it is a good test
  of whether your analysis code is written generally. Notice that nothing in
  `plot_across_sessions` knew how many phases to expect.

### Q13. Say how confident you are

Every bar in this notebook already carries a bootstrap error bar, so you have been
reading uncertainty all along. The next step is to put an error bar on the
**difference**, which is the thing you actually want to claim.

In [ ]:
# --- Q13 -----------------------------------------------------------------------
def compute_difference_ci(values, mask_a, mask_b, n_resamples=1000):
    """The median difference between two groups, with a bootstrap range.

    Resample each group independently, take the difference of the two medians,
    and repeat. If the resulting range does not cross zero, the difference is
    bigger than the noise.
    """
    a = values[mask_a]
    a = a[~np.isnan(a)]
    b = values[mask_b]
    b = b[~np.isnan(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan, np.nan

    differences = np.full(n_resamples, np.nan)
    for k in range(n_resamples):
        resample_a = np.random.choice(a, size=len(a), replace=True)
        resample_b = np.random.choice(b, size=len(b), replace=True)
        differences[k] = np.median(resample_b) - np.median(resample_a)

    return (np.median(b) - np.median(a),
            np.percentile(differences, 2.5), np.percentile(differences, 97.5))


# The claim the anatomy predicts: the CHANGE differs between the two directions.
# Compare "before" against every later phase, because different measures do their
# most interesting thing at different times -- RT during the injection, peak
# velocity after it.
before_name, before_mask = injection_phases[0]
later_phases = injection_phases[1:]

fig, ax = plt.subplots(figsize=(8.5, 4.8))

for k, (group_name, group_mask) in enumerate(DIRECTION_GROUPS):
    x_positions, changes, down, up = [], [], [], []

    for j, (phase_name, phase_mask) in enumerate(later_phases):
        change, low, high = compute_difference_ci(
            saccades["reaction_time"],
            before_mask & group_mask & saccades["detected"],
            phase_mask & group_mask & saccades["detected"])

        x_positions.append(j + (k - 0.5) * 0.18)
        changes.append(change)
        down.append(change - low)
        up.append(high - change)

    ax.errorbar(x_positions, changes, yerr=[down, up], fmt="o", markersize=10,
                capsize=7, linewidth=2, linestyle="none",
                color=["tab:blue", "tab:red"][k], label=group_name)

ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.set_xticks(range(len(later_phases)))
ax.set_xticklabels(["%s to %s" % (before_name, name) for name, mask in later_phases])
ax.set_xlim(-0.5, len(later_phases) - 0.5)
ax.set_ylabel("Change in median reaction time (ms)")
ax.set_title("Q13: the change, with a bootstrap 95% range\n"
             "a bar that does not cross zero is a real change")
ax.legend()
fig.tight_layout()
plt.show()

# TODO 1: for "before to during", does the rightward range sit clear of zero
#         while the leftward one straddles it? That pattern -- one direction
#         changed, the other demonstrably not -- is the strongest form of this
#         result, and it is much stronger than two separate tests.
# TODO 2: run the identical figure for peak velocity by swapping
#         saccades["reaction_time"] for saccades["peak_velocity"]. Which phase
#         shows the effect there? It is not the same one.

### Q14. Does the detector change the conclusion?

Every number in this notebook rests on the settings in `SETTINGS`. A result that
holds across reasonable settings is worth reporting; one that appears only at a
particular threshold is worth being suspicious of.

This is the single most valuable habit in the whole notebook, so here it is drawn
out: re-run the detection at a range of velocity thresholds and watch what happens.

*This cell re-runs the detector several times, so give it a moment.*

In [ ]:
# --- Q14 -----------------------------------------------------------------------
thresholds_to_try = [20, 30, 40, 50, 60, 80]

sweep_rt = {"leftward": [], "rightward": []}
sweep_detected = []

for threshold in thresholds_to_try:
    # Copy the settings and change the one number we are testing.
    trial_settings = dict(SETTINGS)
    trial_settings["velocity_threshold"] = threshold

    found = detect_saccades(smooth_x, smooth_y, eye_speed, speed_time, trial_settings)

    sweep_detected.append(100 * np.sum(found["detected"]) / n_trials)
    for group_name, group_mask in DIRECTION_GROUPS:
        selected = group_mask & found["detected"]
        sweep_rt[group_name].append(np.nanmedian(found["reaction_time"][selected]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

for (group_name, values), colour in zip(sweep_rt.items(), ["tab:blue", "tab:red"]):
    axes[0].plot(thresholds_to_try, values, "o-", color=colour, linewidth=2, label=group_name)
axes[0].axvline(SETTINGS["velocity_threshold"], color="black", linestyle="--",
                linewidth=1.5, label="setting we used")
axes[0].set_xlabel("velocity_threshold (deg/s above baseline)")
axes[0].set_ylabel("Median reaction time (ms)")
axes[0].set_title("Does the RT depend on the setting?")
axes[0].legend()

axes[1].plot(thresholds_to_try, sweep_detected, "o-", color="tab:green", linewidth=2)
axes[1].axvline(SETTINGS["velocity_threshold"], color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("velocity_threshold (deg/s above baseline)")
axes[1].set_ylabel("Saccades detected (%)")
axes[1].set_title("...and how many saccades survive?")

fig.suptitle("Q14: how much does the answer depend on the detector settings?", y=1.03)
fig.tight_layout()
plt.show()

# TODO 1: the RTs shift with the threshold -- of course they do, a higher bar is
#         crossed later. The question is whether the GAP between the two lines
#         stays. Does it?
# TODO 2: do the same sweep for n_contiguous (try 3, 5, 8) and for smooth_span.
# TODO 3: now the real test: repeat the Q13 figure at threshold 20 and at 60.
#         Does the conclusion survive?

---
## Where to look if something breaks

| Symptom | Likely cause |
|---|---|
| A group comes out empty | You tested a float with `==` (`target_angle == 45` never matches — the stored value is `45.00000002`), or you used another session's hemifield codes. Use `target_x < 0`. |
| Everything is `nan` | You used `np.mean` instead of `np.nanmean`, or forgot to pass `saccades["detected"]` as `valid`. |
| `np.isnan` complains about the type | The array holds integers or text. `np.isnan` only works on floats. |
| A bar chart comes out empty | Your two masks probably do not overlap — check `np.sum(mask_a & mask_b)` before plotting. |
| A firing rate looks impossibly high or low | Check which event you aligned to. The same window means different things aligned to the target versus the GO cue. |
| The detection rate collapses | You probably lowered `noise_speed` below the fastest real saccade, which deletes good trials rather than bad ones. |

## Credit

The data was recorded in the OHLab. This notebook is a Python translation of the
MATLAB analysis in `LoadAndVisulizeData_Old.m` and its helper functions; the saccade
detector follows `RT_Old.m` and reproduces its published numbers on the
`Adams102325_FRAC` session.